In [1]:
!pip install fastapi uvicorn "pydantic[email]" httpx pytest nest_asyncio ipytest

In [2]:
from fastapi import FastAPI, APIRouter, Depends, HTTPException, status
from fastapi.middleware.cors import CORSMiddleware

from pydantic import (
    BaseModel,
    EmailStr,
    Field,
    field_validator,
    model_validator,
    ConfigDict
)

from datetime import datetime
from enum import Enum
from typing import List, Optional, Dict
from uuid import UUID, uuid4

In [3]:
class OrderStatus(str, Enum):
    PENDING = "pending"
    CONFIRMED = "confirmed"
    SHIPPED = "shipped"
    DELIVERED = "delivered"
    CANCELLED = "cancelled"


class PaymentMethod(str, Enum):
    CARD = "card"
    UPI = "upi"
    CASH = "cash"
    WALLET = "wallet"


class Address(BaseModel):

    model_config = ConfigDict(
        json_schema_extra={
            "example": {
                "street": "42 MG Road",
                "city": "Dehradun",
                "state": "Uttarakhand",
                "pincode": "248001",
                "country": "India"
            }
        }
    )

    street: str = Field(..., min_length=3)
    city: str
    state: str
    pincode: str = Field(..., pattern=r"^\d{6}$")
    country: str = "India"


class Product(BaseModel):

    id: UUID = Field(default_factory=uuid4)
    name: str = Field(..., min_length=1)
    description: Optional[str] = None
    price: float = Field(..., gt=0)
    sku: str
    in_stock: bool = True

    @field_validator("price")
    @classmethod
    def price_precision(cls, v):
        return round(v, 2)


class OrderItem(BaseModel):

    product: Product
    quantity: int = Field(..., ge=1, le=100)

    discount: float = Field(
        default=0.0,
        ge=0.0,
        le=100.0
    )

    @property
    def line_total(self):
        discounted = self.product.price * (
            1 - self.discount / 100
        )
        return round(discounted * self.quantity, 2)


class Customer(BaseModel):

    id: UUID = Field(default_factory=uuid4)
    name: str = Field(..., min_length=2)
    email: EmailStr

    phone: str = Field(
        ...,
        pattern=r"^\+?[1-9]\d{9,14}$"
    )

    shipping_address: Address
    billing_address: Optional[Address] = None

    @model_validator(mode="after")
    def set_billing_address(self):

        if self.billing_address is None:
            self.billing_address = self.shipping_address

        return self


class OrderCreate(BaseModel):

    customer: Customer

    items: List[OrderItem] = Field(
        ...,
        min_length=1
    )

    payment_method: PaymentMethod
    notes: Optional[str] = None

    @field_validator("items")
    @classmethod
    def items_not_empty(cls, v):

        if not v:
            raise ValueError(
                "Order must contain at least one item."
            )

        return v


class Order(OrderCreate):

    id: UUID = Field(default_factory=uuid4)

    status: OrderStatus = OrderStatus.PENDING

    created_at: datetime = Field(
        default_factory=datetime.utcnow
    )

    updated_at: datetime = Field(
        default_factory=datetime.utcnow
    )


class OrderSummary(BaseModel):

    subtotal: float
    total_discount: float
    grand_total: float
    item_count: int


class OrderResponse(Order):

    summary: OrderSummary

    @classmethod
    def from_order(cls, order):

        items = order.items

        subtotal = sum(
            i.product.price * i.quantity
            for i in items
        )

        total_discount = sum(
            (i.product.price * i.quantity)
            - i.line_total
            for i in items
        )

        grand_total = round(
            subtotal - total_discount,
            2
        )

        return cls(
            **order.model_dump(),
            summary=OrderSummary(
                subtotal=round(subtotal, 2),
                total_discount=round(
                    total_discount,
                    2
                ),
                grand_total=grand_total,
                item_count=sum(
                    i.quantity
                    for i in items
                )
            )
        )

In [4]:
class OrderRepository:

    def __init__(self):
        self._store: Dict[UUID, Order] = {}

    def save(self, order):

        self._store[order.id] = order
        return order

    def get(self, order_id):
        return self._store.get(order_id)

    def list_all(self):
        return list(self._store.values())

    def update_status(
        self,
        order_id,
        status
    ):

        order = self._store.get(order_id)

        if not order:
            return None

        updated = order.model_copy(
            update={"status": status}
        )

        self._store[order_id] = updated

        return updated

    def delete(self, order_id):

        if order_id in self._store:

            del self._store[order_id]
            return True

        return False


_repo = OrderRepository()


def get_repository():
    return _repo

In [5]:
router = APIRouter(
    prefix="/orders",
    tags=["Orders"]
)

In [6]:
@router.post(
    "/",
    response_model=OrderResponse,
    status_code=201
)
def create_order(
    payload: OrderCreate,
    repo: OrderRepository = Depends(
        get_repository
    )
):

    order = Order(
        **payload.model_dump()
    )

    saved = repo.save(order)

    return OrderResponse.from_order(
        saved
    )

In [7]:
@router.get(
    "/",
    response_model=List[OrderResponse]
)
def list_orders(
    repo: OrderRepository = Depends(
        get_repository
    )
):

    return [
        OrderResponse.from_order(o)
        for o in repo.list_all()
    ]

In [8]:
@router.get(
    "/{order_id}",
    response_model=OrderResponse
)
def get_order(
    order_id: UUID,
    repo: OrderRepository = Depends(
        get_repository
    )
):

    order = repo.get(order_id)

    if not order:

        raise HTTPException(
            status_code=404,
            detail="Order not found"
        )

    return OrderResponse.from_order(
        order
    )

In [9]:
@router.patch(
    "/{order_id}/status",
    response_model=OrderResponse
)
def update_order_status(
    order_id: UUID,
    new_status: OrderStatus,
    repo: OrderRepository = Depends(
        get_repository
    )
):

    order = repo.update_status(
        order_id,
        new_status
    )

    if not order:

        raise HTTPException(
            status_code=404,
            detail="Order not found"
        )

    return OrderResponse.from_order(
        order
    )

In [10]:
app = FastAPI(
    title="Order Management API"
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"]
)

app.include_router(router)


@app.get("/")
def health():

    return {
        "status": "ok",
        "message": "Order Management API is running"
    }

In [11]:
import asyncio
import nest_asyncio
import uvicorn

nest_asyncio.apply()

config = uvicorn.Config(
    app,
    host="0.0.0.0",
    port=8000,
    loop="asyncio"
)

server = uvicorn.Server(config)

asyncio.create_task(
    server.serve()
)

<Task pending name='Task-1' coro=<Server.serve() running at c:\Users\Administrator\Learnings\ai-training\.venv\Lib\site-packages\uvicorn\server.py:77>>

INFO:     Started server process [11496]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [12]:
!pip install ipytest pytest -q

import ipytest
ipytest.autoconfig()

In [13]:
import pytest
from uuid import uuid4
from fastapi.testclient import TestClient

client = TestClient(app)

ORDER_PAYLOAD = {
    "customer": {
        "name": "Aarav Sharma",
        "email": "aarav@example.com",
        "phone": "+919876543210",
        "shipping_address": {
            "street": "42 MG Road",
            "city": "Dehradun",
            "state": "Uttarakhand",
            "pincode": "248001",
            "country": "India"
        }
    },
    "items": [
        {
            "product": {
                "name": "Wireless Headphones",
                "price": 2999.0,
                "sku": "WH-001"
            },
            "quantity": 2,
            "discount": 10
        }
    ],
    "payment_method": "upi"
}


def test_invalid_pincode():

    bad = ORDER_PAYLOAD.copy()

    bad["customer"][
        "shipping_address"
    ]["pincode"] = "12AB"

    r = client.post(
        "/orders/",
        json=bad
    )

    assert r.status_code == 422


In [14]:
def test_health():
    r = client.get("/")
    assert r.status_code == 200

def test_create_order_returns_summary():
    r = client.post("/orders/", json=ORDER_PAYLOAD)
    assert r.status_code == 201

def test_list_orders():
    r = client.get("/orders/")
    assert r.status_code == 200

In [ ]:
# ipytest.run("-v")

In [15]:
#TASK 1 - Order Filtering by Status
#TASK 2 - Pagination + Sorting

from typing import List, Optional, Literal
from fastapi import Query

@router.get(
    "/",
    response_model=List[OrderResponse],
    summary="List all orders",
)
def list_orders(
    status: Optional[OrderStatus] = None,
    skip: int = Query(0, ge=0),
    limit: int = Query(10, ge=1, le=100),
    sort_by: Literal["created_at", "grand_total"] = "created_at",
    repo: OrderRepository = Depends(get_repository),
) -> List[OrderResponse]:

    orders = repo.list_all()

    # Status Filter
    if status:
        orders = [o for o in orders if o.status == status]

    # Sorting
    if sort_by == "grand_total":
        orders.sort(
            key=lambda o: OrderResponse.from_order(
                o
            ).summary.grand_total,
            reverse=True,
        )
    else:
        orders.sort(
            key=lambda o: o.created_at,
            reverse=True,
        )

    # Pagination
    orders = orders[skip : skip + limit]

    return [OrderResponse.from_order(o) for o in orders]